# ML_A5 — Ensemble Models📝 **Modalidad: Trabajo autónomo — practica a tu ritmo.****Versión:** 2025-1 | **Modificado:** 2026-04-24---

## 🎯 Contexto del Problema**Dataset:** `digits` de sklearn- **Tamaño:** 1797 imágenes- **Features:** 64 (pixeles de imágenes 8×8)- **Clases:** 10 (dígitos 0–9)- **Tarea:** Clasificación multiclase**Pregunta central:**> ¿Pueden modelos simples combinados (ensambles) superar a un modelo complejo individual?**Restricciones:**- ❌ No usar redes neuronales- ✅ Usar ensambles (bagging, boosting, stacking)- ✅ Comparar con baseline (Decision Tree simple)---

## Setup (NO MODIFICAR)

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.datasets import load_digitsfrom sklearn.model_selection import train_test_split, cross_val_score, learning_curvefrom sklearn.preprocessing import StandardScalerfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import (    RandomForestClassifier,    GradientBoostingClassifier,    StackingClassifier)from sklearn.linear_model import LogisticRegressionfrom xgboost import XGBClassifierfrom sklearn.metrics import (    accuracy_score,    classification_report,    confusion_matrix,    roc_auc_score)import timeimport warningswarnings.filterwarnings('ignore')# Configuración visualplt.style.use('seaborn-v0_8-darkgrid')sns.set_palette("husl")# RANDOM STATERANDOM_STATE = 42np.random.seed(RANDOM_STATE)print("✅ Imports completos")

In [ ]:
# Cargar datasetdigits = load_digits()X, y = digits.data, digits.targetprint(f"✅ Dataset cargado:")print(f"   Forma: {X.shape}")print(f"   Clases: {np.unique(y)}")print(f"   Instancias por clase: {np.bincount(y)}")

In [ ]:
# Train-test split estratificado (80-20)X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)print(f"✅ Train-test split:")print(f"   Train: {X_train.shape[0]} samples")print(f"   Test: {X_test.shape[0]} samples")print(f"   Distribución train: {np.bincount(y_train)}")

## Sección 1 — EDA y Preprocesamiento**Objetivo:** Entender la estructura del dataset y preparar los datos para los modelos.

In [ ]:
# Visualizar ejemplos de cada dígitofig, axes = plt.subplots(2, 5, figsize=(12, 5))axes = axes.ravel()for digit in range(10):    idx = np.where(y_train == digit)[0][0]    axes[digit].imshow(X_train[idx].reshape(8, 8), cmap='gray')    axes[digit].set_title(f'Dígito {digit}')    axes[digit].axis('off')plt.tight_layout()plt.savefig('digits_examples.png', dpi=100, bbox_inches='tight')plt.show()print("✅ Ejemplos visualizados")

In [ ]:
# Verificación de valores nulosprint(f"Valores nulos en X_train: {np.isnan(X_train).sum()}")print(f"Valores nulos en y_train: {np.isnan(y_train).sum()}")# Estadísticas básicasprint(f"\nRango de valores en X_train: [{X_train.min():.2f}, {X_train.max():.2f}]")print(f"Media: {X_train.mean():.2f}, Desv. Est.: {X_train.std():.2f}")

In [ ]:
# Normalizar con StandardScalerscaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)print(f"✅ Datos normalizados:")print(f"   Media train escalado: {X_train_scaled.mean():.6f}")print(f"   Desv. Est. train escalado: {X_train_scaled.std():.6f}")print(f"   Media test escalado: {X_test_scaled.mean():.6f}")

In [ ]:
# Tests de sanidad Sección 1assert X_train_scaled.shape == (1437, 64), "Error en shape de X_train_scaled"assert X_test_scaled.shape == (360, 64), "Error en shape de X_test_scaled"assert np.abs(X_train_scaled.mean()) < 0.01, "Media no cercana a 0"assert np.abs(X_train_scaled.std() - 1.0) < 0.01, "Desv. Est. no cercana a 1"assert y_train.shape[0] == 1437, "Error en y_train"print("✅ Sanity checks Sección 1: PASSED")

## Sección 2 — Modelo Base**Objetivo:** Establecer un baseline con un árbol de decisión simple para comparar con ensambles.

In [ ]:
# Entrenar Decision Tree baselinedt_baseline = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)dt_baseline.fit(X_train_scaled, y_train)y_pred_dt = dt_baseline.predict(X_test_scaled)dt_accuracy = accuracy_score(y_test, y_pred_dt)print(f"✅ Decision Tree (max_depth=5) entrenado")print(f"   Accuracy en test: {dt_accuracy:.4f}")

In [ ]:
# Reporte de clasificaciónprint("Classification Report (Decision Tree):")print(classification_report(y_test, y_pred_dt, digits=3))

In [ ]:
# Confusion matrixcm_dt = confusion_matrix(y_test, y_pred_dt)plt.figure(figsize=(8, 6))sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', cbar=True,            xticklabels=range(10), yticklabels=range(10))plt.title('Confusion Matrix — Decision Tree Baseline')plt.xlabel('Predicción')plt.ylabel('Verdadero')plt.tight_layout()plt.savefig('cm_baseline.png', dpi=100, bbox_inches='tight')plt.show()

In [ ]:
# Tests de sanidad Sección 2assert dt_accuracy > 0.5, "Accuracy baseline demasiado bajo"assert cm_dt.shape == (10, 10), "Confusion matrix con shape incorrecto"assert cm_dt.sum() == len(y_test), "Sum de CM no coincide con test size"print("✅ Sanity checks Sección 2: PASSED")

## Sección 3 — Comparación de Ensembles**Objetivo:** Entrenar y comparar tres ensambles: Random Forest, Gradient Boosting, XGBoost.### Parte Compartida

In [ ]:
# TODO 1: Entrenar Random Forestprint("Entrenando Random Forest...")rf_model = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=RANDOM_STATE)rf_model.fit(X_train_scaled, y_train)y_pred_rf = rf_model.predict(X_test_scaled)rf_accuracy = accuracy_score(y_test, y_pred_rf)print(f"✅ Random Forest entrenado")print(f"   Accuracy en test: {rf_accuracy:.4f}")

In [ ]:
# TODO 2: Entrenar Gradient Boostingprint("Entrenando Gradient Boosting...")gb_model = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_STATE)gb_model.fit(X_train_scaled, y_train)y_pred_gb = gb_model.predict(X_test_scaled)gb_accuracy = accuracy_score(y_test, y_pred_gb)print(f"✅ Gradient Boosting entrenado")print(f"   Accuracy en test: {gb_accuracy:.4f}")

In [ ]:
# TODO 3: Entrenar XGBoostprint("Entrenando XGBoost...")xgb_model = XGBClassifier(    n_estimators=100,    max_depth=4,    use_label_encoder=False,    eval_metric='mlogloss',    random_state=RANDOM_STATE,    verbose=0)xgb_model.fit(X_train_scaled, y_train)y_pred_xgb = xgb_model.predict(X_test_scaled)xgb_accuracy = accuracy_score(y_test, y_pred_xgb)print(f"✅ XGBoost entrenado")print(f"   Accuracy en test: {xgb_accuracy:.4f}")

In [ ]:
# TODO 4: Tabla comparativa con cross_val_scoreprint("Calculando cross-validation (cv=5)...")cv_results = {}models = {    'Decision Tree (baseline)': dt_baseline,    'Random Forest': rf_model,    'Gradient Boosting': gb_model,    'XGBoost': xgb_model}for name, model in models.items():    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')    cv_results[name] = {        'mean': cv_scores.mean(),        'std': cv_scores.std(),        'test_accuracy': accuracy_score(y_test, model.predict(X_test_scaled))    }# Crear DataFramecomparison_df = pd.DataFrame(cv_results).Tcomparison_df = comparison_df.round(4)comparison_df.columns = ['CV Mean', 'CV Std', 'Test Accuracy']print("\n" + "="*60)print("Tabla Comparativa de Ensambles")print("="*60)print(comparison_df)print("="*60)

In [ ]:
# TODO 5: Medir tiempo de entrenamientoimport timetraining_times = {}# Decision Treet0 = time.time()dt_baseline.fit(X_train_scaled, y_train)training_times['Decision Tree'] = time.time() - t0# Random Forestt0 = time.time()rf_model.fit(X_train_scaled, y_train)training_times['Random Forest'] = time.time() - t0# Gradient Boostingt0 = time.time()gb_model.fit(X_train_scaled, y_train)training_times['Gradient Boosting'] = time.time() - t0# XGBoostt0 = time.time()xgb_model.fit(X_train_scaled, y_train)training_times['XGBoost'] = time.time() - t0# Accuracy por modeloaccuracies = {    'Decision Tree': dt_accuracy,    'Random Forest': rf_accuracy,    'Gradient Boosting': gb_accuracy,    'XGBoost': xgb_accuracy}# Tabla de tiempostimes_df = pd.DataFrame({    'Modelo': list(training_times.keys()),    'Tiempo (s)': list(training_times.values()),    'Accuracy': list(accuracies.values())}).round(4)print("\nTiempos de Entrenamiento:")print(times_df)

In [ ]:
# Gráfico: Accuracy vs Tiempofig, ax = plt.subplots(figsize=(10, 6))colors = ['#2E86C1', '#E74C3C', '#F39C12', '#27AE60']models_list = list(training_times.keys())times_list = [training_times[m] for m in models_list]acc_list = [accuracies[m] for m in models_list]bars = ax.barh(models_list, times_list, color=colors, alpha=0.8, edgecolor='black')# Agregar valores de accuracy en las barrasfor i, (t, a) in enumerate(zip(times_list, acc_list)):    ax.text(t + 0.01, i, f'{a:.4f}', va='center', fontweight='bold', fontsize=10)ax.set_xlabel('Tiempo de Entrenamiento (segundos)', fontsize=12, fontweight='bold')ax.set_title('Comparación: Tiempo de Entrenamiento vs Accuracy', fontsize=13, fontweight='bold')ax.grid(axis='x', alpha=0.3)plt.tight_layout()plt.savefig('training_time_vs_accuracy.png', dpi=100, bbox_inches='tight')plt.show()print("✅ Gráfico de tiempo vs accuracy generado")

In [ ]:
# TODO 5 [PhD]: Curvas de aprendizaje para RF y GBMtrain_sizes = np.linspace(0.1, 1.0, 10)fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Random Foresttrain_sizes_abs, train_scores_rf, val_scores_rf = learning_curve(    RandomForestClassifier(n_estimators=100, max_depth=None, random_state=RANDOM_STATE),    X_train_scaled, y_train,    train_sizes=train_sizes,    cv=5,    scoring='accuracy',    n_jobs=-1)train_mean_rf = train_scores_rf.mean(axis=1)train_std_rf = train_scores_rf.std(axis=1)val_mean_rf = val_scores_rf.mean(axis=1)val_std_rf = val_scores_rf.std(axis=1)axes[0].plot(train_sizes_abs, train_mean_rf, 'o-', label='Train', linewidth=2, markersize=6)axes[0].fill_between(train_sizes_abs, train_mean_rf - train_std_rf, train_mean_rf + train_std_rf, alpha=0.2)axes[0].plot(train_sizes_abs, val_mean_rf, 'o-', label='Validación', linewidth=2, markersize=6)axes[0].fill_between(train_sizes_abs, val_mean_rf - val_std_rf, val_mean_rf + val_std_rf, alpha=0.2)axes[0].set_title('Random Forest — Curva de Aprendizaje', fontsize=12, fontweight='bold')axes[0].set_xlabel('Tamaño del Conjunto de Entrenamiento')axes[0].set_ylabel('Accuracy')axes[0].legend()axes[0].grid(alpha=0.3)# Gradient Boostingtrain_sizes_abs, train_scores_gb, val_scores_gb = learning_curve(    GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_STATE),    X_train_scaled, y_train,    train_sizes=train_sizes,    cv=5,    scoring='accuracy',    n_jobs=-1)train_mean_gb = train_scores_gb.mean(axis=1)train_std_gb = train_scores_gb.std(axis=1)val_mean_gb = val_scores_gb.mean(axis=1)val_std_gb = val_scores_gb.std(axis=1)axes[1].plot(train_sizes_abs, train_mean_gb, 'o-', label='Train', linewidth=2, markersize=6)axes[1].fill_between(train_sizes_abs, train_mean_gb - train_std_gb, train_mean_gb + train_std_gb, alpha=0.2)axes[1].plot(train_sizes_abs, val_mean_gb, 'o-', label='Validación', linewidth=2, markersize=6)axes[1].fill_between(train_sizes_abs, val_mean_gb - val_std_gb, val_mean_gb + val_std_gb, alpha=0.2)axes[1].set_title('Gradient Boosting — Curva de Aprendizaje', fontsize=12, fontweight='bold')axes[1].set_xlabel('Tamaño del Conjunto de Entrenamiento')axes[1].set_ylabel('Accuracy')axes[1].legend()axes[1].grid(alpha=0.3)plt.tight_layout()plt.savefig('learning_curves.png', dpi=100, bbox_inches='tight')plt.show()# Análisisprint("\n📊 Análisis de Curvas de Aprendizaje:")print(f"\nRandom Forest:")print(f"  Brecha train-test final: {train_mean_rf[-1] - val_mean_rf[-1]:.4f}")print(f"  Accuracy validación final: {val_mean_rf[-1]:.4f}")print(f"\nGradient Boosting:")print(f"  Brecha train-test final: {train_mean_gb[-1] - val_mean_gb[-1]:.4f}")print(f"  Accuracy validación final: {val_mean_gb[-1]:.4f}")print("\n💡 Interpretación:")print("  - Mayor brecha → Mayor varianza (RF)")print("  - Menor brecha → Mejor regularización (GB)")

In [ ]:
# Tests de sanidad Sección 3assert rf_accuracy > dt_accuracy, "RF no supera baseline"assert gb_accuracy > dt_accuracy, "GB no supera baseline"assert xgb_accuracy > dt_accuracy, "XGB no supera baseline"assert len(comparison_df) == 4, "Tabla comparativa incompleta"assert all(comparison_df['Test Accuracy'] > 0.6), "Algún modelo tiene accuracy muy bajo"print("✅ Sanity checks Sección 3: PASSED")

## Sección 4 — Análisis de Feature Importance**Objetivo:** Entender qué pixeles (features) son más informativos para clasificar dígitos.

In [ ]:
# TODO 6: Feature importance del Random Forest como heatmap 8x8feature_importance_rf = rf_model.feature_importances_importance_grid = feature_importance_rf.reshape(8, 8)fig, axes = plt.subplots(1, 2, figsize=(12, 5))# Heatmap de importanciasns.heatmap(importance_grid, cmap='YlOrRd', ax=axes[0], cbar_kws={'label': 'Importancia'})axes[0].set_title('Feature Importance — Random Forest\n(Visualización 8x8)', fontsize=12, fontweight='bold')axes[0].set_xlabel('Columna (píxel)')axes[0].set_ylabel('Fila (píxel)')# Top 20 featurestop_20_indices = np.argsort(feature_importance_rf)[-20:]top_20_importance = feature_importance_rf[top_20_indices]axes[1].barh(range(20), top_20_importance, color='steelblue', edgecolor='black')axes[1].set_yticks(range(20))axes[1].set_yticklabels([f'Pixel {i}' for i in top_20_indices], fontsize=9)axes[1].set_xlabel('Importancia', fontweight='bold')axes[1].set_title('Top 20 Features Más Importantes', fontsize=12, fontweight='bold')axes[1].invert_yaxis()axes[1].grid(axis='x', alpha=0.3)plt.tight_layout()plt.savefig('feature_importance_rf.png', dpi=100, bbox_inches='tight')plt.show()print("✅ Feature importance visualizado")

In [ ]:
# TODO 7: Comparar importancias RF vs GBfeature_importance_gb = gb_model.feature_importances_feature_importance_xgb = xgb_model.feature_importances_# Correlación entre importanciascorr_rf_gb = np.corrcoef(feature_importance_rf, feature_importance_gb)[0, 1]corr_rf_xgb = np.corrcoef(feature_importance_rf, feature_importance_xgb)[0, 1]print("Correlación de Feature Importances:")print(f"  RF vs GB: {corr_rf_gb:.4f}")print(f"  RF vs XGB: {corr_rf_xgb:.4f}")# Visualizar heatmaps lado a ladofig, axes = plt.subplots(1, 3, figsize=(15, 4))sns.heatmap(feature_importance_rf.reshape(8, 8), cmap='YlOrRd', ax=axes[0],            cbar_kws={'label': 'Importancia'})axes[0].set_title('Random Forest', fontsize=11, fontweight='bold')sns.heatmap(feature_importance_gb.reshape(8, 8), cmap='YlOrRd', ax=axes[1],            cbar_kws={'label': 'Importancia'})axes[1].set_title('Gradient Boosting', fontsize=11, fontweight='bold')sns.heatmap(feature_importance_xgb.reshape(8, 8), cmap='YlOrRd', ax=axes[2],            cbar_kws={'label': 'Importancia'})axes[2].set_title('XGBoost', fontsize=11, fontweight='bold')plt.tight_layout()plt.savefig('feature_importance_comparison.png', dpi=100, bbox_inches='tight')plt.show()print("\n✅ Comparación de importancias visualizada")

In [ ]:
# TODO 7 [PhD]: Comparar con PCAfrom sklearn.decomposition import PCA# Entrenar PCApca = PCA(n_components=20)pca.fit(X_train_scaled)# Top 20 componentespca_loadings_sum = np.abs(pca.components_).sum(axis=0)fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Top 20 features por importancia RFtop_indices_rf = np.argsort(feature_importance_rf)[-20:]axes[0].barh(range(20), feature_importance_rf[top_indices_rf], color='#2E86C1', edgecolor='black')axes[0].set_yticks(range(20))axes[0].set_yticklabels([f'Pixel {i}' for i in top_indices_rf], fontsize=9)axes[0].set_xlabel('Importancia (RF)', fontweight='bold')axes[0].set_title('Random Forest — Top 20 Features', fontsize=12, fontweight='bold')axes[0].invert_yaxis()axes[0].grid(axis='x', alpha=0.3)# Top 20 features por PCA loadingstop_indices_pca = np.argsort(pca_loadings_sum)[-20:]axes[1].barh(range(20), pca_loadings_sum[top_indices_pca], color='#B7950B', edgecolor='black')axes[1].set_yticks(range(20))axes[1].set_yticklabels([f'Pixel {i}' for i in top_indices_pca], fontsize=9)axes[1].set_xlabel('|Loadings| Sum (PCA)', fontweight='bold')axes[1].set_title('PCA — Top 20 Features', fontsize=12, fontweight='bold')axes[1].invert_yaxis()axes[1].grid(axis='x', alpha=0.3)plt.tight_layout()plt.savefig('rf_vs_pca_features.png', dpi=100, bbox_inches='tight')plt.show()# Análisisprint("\n📊 Análisis RF vs PCA:")print(f"Varianza explicada por primeros 20 PCs: {pca.explained_variance_ratio_[:20].sum():.4f}")print(f"\nSuperposición top-5 features RF vs PCA:")top_5_rf = set(np.argsort(feature_importance_rf)[-5:])top_5_pca = set(np.argsort(pca_loadings_sum)[-5:])overlap = len(top_5_rf & top_5_pca)print(f"  Features en común: {overlap}/5")

In [ ]:
# Tests de sanidad Sección 4assert len(feature_importance_rf) == 64, "Feature importance RF con tamaño incorrecto"assert len(feature_importance_gb) == 64, "Feature importance GB con tamaño incorrecto"assert feature_importance_rf.sum() > 0, "Importancias RF no positivas"assert feature_importance_gb.sum() > 0, "Importancias GB no positivas"print("✅ Sanity checks Sección 4: PASSED")

## Sección 5 — Stacking (Ensambles Avanzados)**Objetivo:** Combinar múltiples modelos base usando un meta-learner para mejorar performance.### Concepto: Stacking- **Estimadores base:** Entrenan en los datos originales- **Meta-learner:** Aprende a combinar predicciones de los base models- **Out-of-fold:** Usa validación cruzada para evitar data leakage

In [ ]:
# TODO 8: Implementar StackingClassifierfrom sklearn.ensemble import StackingClassifier# Definir estimadores basebase_learners = [    ('rf', RandomForestClassifier(n_estimators=50, max_depth=None, random_state=RANDOM_STATE)),    ('gb', GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=RANDOM_STATE)),    ('dt', DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))]# Meta-learnermeta_learner = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)# StackingClassifier con cv=5stacking_model = StackingClassifier(    estimators=base_learners,    final_estimator=meta_learner,    cv=5)print("✅ StackingClassifier creado")print(f"   Estimadores base: {len(base_learners)}")print(f"   Meta-learner: LogisticRegression")print(f"   CV folds: 5")

In [ ]:
# Entrenar stacking modelprint("Entrenando StackingClassifier (esto puede tomar un tiempo)...")stacking_model.fit(X_train_scaled, y_train)# Evaluary_pred_stack = stacking_model.predict(X_test_scaled)stack_accuracy = accuracy_score(y_test, y_pred_stack)print(f"✅ StackingClassifier entrenado")print(f"   Accuracy en test: {stack_accuracy:.4f}")

In [ ]:
# Cross-validation del stackingcv_scores_stack = cross_val_score(stacking_model, X_train_scaled, y_train, cv=5, scoring='accuracy')print(f"\nCross-Validation (cv=5):")print(f"  Mean: {cv_scores_stack.mean():.4f}")print(f"  Std: {cv_scores_stack.std():.4f}")# Tabla comparativa actualizadafinal_comparison = pd.DataFrame({    'CV Mean': [        cross_val_score(dt_baseline, X_train_scaled, y_train, cv=5).mean(),        cross_val_score(rf_model, X_train_scaled, y_train, cv=5).mean(),        cross_val_score(gb_model, X_train_scaled, y_train, cv=5).mean(),        cross_val_score(xgb_model, X_train_scaled, y_train, cv=5).mean(),        cv_scores_stack.mean()    ],    'CV Std': [        cross_val_score(dt_baseline, X_train_scaled, y_train, cv=5).std(),        cross_val_score(rf_model, X_train_scaled, y_train, cv=5).std(),        cross_val_score(gb_model, X_train_scaled, y_train, cv=5).std(),        cross_val_score(xgb_model, X_train_scaled, y_train, cv=5).std(),        cv_scores_stack.std()    ],    'Test Accuracy': [dt_accuracy, rf_accuracy, gb_accuracy, xgb_accuracy, stack_accuracy]}, index=['Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost', 'Stacking'])print("\n" + "="*70)print("TABLA FINAL: Todos los Modelos")print("="*70)print(final_comparison.round(4))print("="*70)

In [ ]:
# TODO 9 [PhD]: Ablation study — Stacking sin DecisionTreebase_learners_no_dt = [    ('rf', RandomForestClassifier(n_estimators=50, max_depth=None, random_state=RANDOM_STATE)),    ('gb', GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=RANDOM_STATE))]stacking_no_dt = StackingClassifier(    estimators=base_learners_no_dt,    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),    cv=5)print("Entrenando stacking SIN DecisionTree...")stacking_no_dt.fit(X_train_scaled, y_train)y_pred_stack_no_dt = stacking_no_dt.predict(X_test_scaled)stack_no_dt_accuracy = accuracy_score(y_test, y_pred_stack_no_dt)print(f"✅ Stacking sin DT entrenado")print(f"   Accuracy: {stack_no_dt_accuracy:.4f}")# Comparaciónprint("\n" + "="*70)print("Ablation Study: Impacto del DecisionTree en Stacking")print("="*70)comparison_stack = pd.DataFrame({    'Modelo': ['Stacking (RF + GB + DT)', 'Stacking (RF + GB)'],    'Estimadores Base': [3, 2],    'Test Accuracy': [stack_accuracy, stack_no_dt_accuracy],    'Diferencia': [0.0, stack_accuracy - stack_no_dt_accuracy]})print(comparison_stack)print("="*70)if stack_accuracy > stack_no_dt_accuracy:    print(f"\n💡 Conclusión: Agregar DT MEJORA el stack en {(stack_accuracy - stack_no_dt_accuracy):.4f}")else:    print(f"\n💡 Conclusión: Agregar DT EMPEORA el stack en {(stack_no_dt_accuracy - stack_accuracy):.4f}")print("\nJustificación teórica:")print("  - Aunque DT es débil, aporta DIVERSIDAD al ensemble")print("  - El meta-learner asigna pesos bajos a predicciones ruidosas")print("  - En general, diversidad > calidad individual en ensambles")

In [ ]:
# Tests de sanidad Sección 5assert stack_accuracy > 0.5, "Accuracy stacking demasiado bajo"assert stack_accuracy >= dt_accuracy, "Stacking no supera baseline"assert final_comparison.shape == (5, 3), "Tabla comparativa incompleta"print("✅ Sanity checks Sección 5: PASSED")

## Sección 6 — AutoevaluaciónAntes de terminar, responde estas preguntas:### Checklist General (Todas las Audiencias)- [ ] Cargué correctamente el dataset `digits`- [ ] Dividí los datos 80/20 y los normalicé- [ ] Entrené todos los modelos base sin errores- [ ] Las métricas de todos los modelos están calculadas- [ ] Entiendo por qué el ensemble supera al árbol individual- [ ] Visualicé feature importances e interpretó los resultados- [ ] Implementé stacking correctamente con out-of-fold### Checklist 🔵 Pregrado- [ ] Identifico el modelo con mejor relación accuracy/tiempo- [ ] Entiendo la diferencia entre bagging y boosting- [ ] Puedo explicar por qué Random Forest reduce varianza- [ ] Reconozco cuáles features (pixeles) son informativos### Checklist 🟡 Doctorado- [ ] Interpreté correctamente las curvas de aprendizaje- [ ] Diagnóstico si el problema es sesgo o varianza- [ ] Comparé feature importances con componentes PCA- [ ] Justifiqué teóricamente el ablation study del stacking- [ ] Entiendo el riesgo de data leakage en stacking y su mitigación### Reflexión Final**Escribe aquí tu respuesta (máximo 200 palabras):**1. ¿Qué fue lo más difícil de entender?2. ¿Qué concepto queda sin claridad completa?3. ¿Cuál fue tu principal aprendizaje?*[Espacio para tu respuesta]*

## Tests de Sanidad Globales (NO MODIFICAR)Verificación final: Todos los modelos entrenados y evaluados correctamente.

In [ ]:
# Tests de sanidad globalesprint("="*70)print("SANITY CHECKS GLOBALES")print("="*70)# Shapesassert X_train_scaled.shape == (1437, 64), "Error en X_train_scaled"assert X_test_scaled.shape == (360, 64), "Error en X_test_scaled"assert y_train.shape == (1437,), "Error en y_train"assert y_test.shape == (360,), "Error en y_test"print("✅ Shapes correctos")# Rangos de accuracyall_accuracies = [dt_accuracy, rf_accuracy, gb_accuracy, xgb_accuracy, stack_accuracy]assert all(0 <= acc <= 1 for acc in all_accuracies), "Algún accuracy fuera de rango [0, 1]"assert all(acc > 0.6 for acc in all_accuracies), "Algún accuracy muy bajo (< 0.6)"print("✅ Rangos de accuracy OK")# No leakage: test accuracy debe ser cercana a CV scorefor model_name, model in [('RF', rf_model), ('GB', gb_model), ('XGB', xgb_model)]:    cv_mean = cross_val_score(model, X_train_scaled, y_train, cv=5).mean()    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))    assert abs(cv_mean - test_acc) < 0.15, f"Posible data leakage en {model_name}"print("✅ No hay evidencia de data leakage")# Stacking supera baselineassert stack_accuracy > dt_accuracy, "Stacking no mejora baseline"print("✅ Stacking supera Decision Tree baseline")# Ensemble mejor que individualassert max(rf_accuracy, gb_accuracy, xgb_accuracy) > dt_accuracy, "Ensambles no mejoran baseline"print("✅ Ensambles mejoran baseline")print("="*70)print("✅ ¡TODOS LOS SANITY CHECKS PASARON\!")print("="*70)print("\n📋 Resumen Final:")print(f"  Baseline (DT): {dt_accuracy:.4f}")print(f"  Random Forest: {rf_accuracy:.4f}")print(f"  Gradient Boosting: {gb_accuracy:.4f}")print(f"  XGBoost: {xgb_accuracy:.4f}")print(f"  Stacking: {stack_accuracy:.4f}")print(f"\nMejora respecto a baseline: +{max(all_accuracies) - dt_accuracy:.4f}")

---## Nota para Prueba EscritaEste notebook es **trabajo autónomo**. La calificación final viene de una **prueba escrita en papel** que evalúa conceptos teóricos.**Conceptos clave que debes dominar:**1. **Bias-Varianza en Ensambles:** ¿Por qué Bagging reduce varianza? ¿Por qué Boosting reduce sesgo?2. **Diversidad:** ¿Por qué es crucial? ¿Qué ocurre cuando todos cometen los mismos errores?3. **Boosting:** ¿Cómo se actualizan los pesos? ¿Qué sucede si accuracy < 0.5?4. **Stacking:** ¿Qué es out-of-fold? ¿Por qué evita data leakage?5. **OOB Error:** ¿Qué es? ¿Buena estimación del test error en RF?**Errores comunes a evitar:**- ❌ Usar test set para elegir modelo- ❌ No normalizar features- ❌ Confundir OOB con cross-validation- ❌ Olvidar que Boosting requiere weak learners- ❌ No estratificar en train-test split---